In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train_df = pd.read_parquet('train_final.parquet')
test_df = pd.read_parquet('test_final.parquet')
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,5008159,7,5,417,11632,417,0,59.571429,157.611185,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
1,80,5004971,7,6,381,11632,381,0,54.428571,144.004464,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
2,80,5005677,8,6,325,11632,325,0,40.625000,114.904852,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
3,80,5005163,8,5,537,11632,537,0,67.125000,189.858171,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye
4,80,5006562,8,6,308,11632,308,0,38.500000,108.894444,...,32,0.0,0.0,0,0,0.0,0.0,0,0,DoS GoldenEye


In [3]:
from utils import handle_values
train_df = handle_values(train_df.copy())
test_df = handle_values(test_df.copy())

In [4]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df[' Label']= encoder.fit_transform(train_df[' Label'])
train_df[' Label'].value_counts()

test_df[' Label']= encoder.transform(test_df[' Label'])
test_df[' Label'].value_counts()

 Label
4     6000
0     6000
2     6000
9     6000
3     2059
1     1966
7     1588
10    1179
6     1159
5     1100
8      360
11     301
Name: count, dtype: int64

In [5]:
X_train = train_df.drop(' Label',axis=1)
y_train = train_df[' Label']
X_test = test_df.drop(' Label',axis=1)
y_test = test_df[' Label']

In [6]:
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

In [7]:
X_train =X_train.drop(to_drop,axis=1)
X_test =X_test.drop(to_drop,axis=1)

In [8]:
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
X_train_np = X_train.values
X_test_np = X_test.values


imputer = SimpleImputer(strategy='constant',fill_value=0)
scaler = RobustScaler()


X_train_np = imputer.fit_transform(X_train_np)
X_train_np = scaler.fit_transform(X_train_np)


X_test_np = imputer.transform(X_test_np)
X_test_np = scaler.transform(X_test_np)


train_final = pd.DataFrame(
    X_train_np, 
    columns=X_train.columns
)
test_final = pd.DataFrame(
    X_test_np, 
    columns=X_test.columns
)

In [9]:
train_final[' Label'] = train_df[' Label']
test_final[' Label'] = test_df[' Label']

In [10]:
train_final.shape

(148278, 45)

In [11]:
y_train.value_counts()

 Label
9     31786
0     30000
4     30000
2     25605
3      8234
7      6350
10     4718
6      4637
5      4399
11     1206
1       983
8       360
Name: count, dtype: int64

In [12]:
import lightgbm as lgb
lgbm_gpu = lgb.LGBMClassifier(
    objective='multiclass', 
    num_class=len(np.unique(y_train)),
    n_estimators=100,            
    device='cpu',           
    n_jobs=-1,                   
    random_state=42,
    min_child_samples=1,
    max_depth=4
)

print("Starting LightGBM training for feature importance using the GPU...")
lgbm_gpu.fit(X_train_np, y_train)
print("Training complete and significantly faster!")

Starting LightGBM training for feature importance using the GPU...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022046 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6421
[LightGBM] [Info] Number of data points in the train set: 148278, number of used features: 36
[LightGBM] [Info] Start training from score -1.597892
[LightGBM] [Info] Start training from score -5.016235
[LightGBM] [Info] Start training from score -1.756301
[LightGBM] [Info] Start training from score -2.890817
[LightGBM] [Info] Start training from score -1.597892
[LightGBM] [Info] Start training from score -3.517712
[LightGBM] [Info] Start training from score -3.465021
[LightGBM] [Info] Start training from score -3.150634
[LightGBM] [Info] Start training from score -6.020740
[LightGBM] [Info] Start training from score -1.540063
[LightGBM] [Info] Start training from s

In [13]:

# --- C. Get Importance Scores ---
feature_importances = pd.Series(lgbm_gpu.feature_importances_, index=X_train.columns)
feature_importances

 Destination Port              1072
 Flow Duration                  649
 Total Fwd Packets              680
Total Length of Fwd Packets     868
 Fwd Packet Length Max          481
 Fwd Packet Length Min          103
 Fwd Packet Length Mean         462
Bwd Packet Length Max           515
 Bwd Packet Length Min          122
Flow Bytes/s                    381
 Flow Packets/s                 285
 Flow IAT Mean                  220
 Flow IAT Std                   342
 Flow IAT Min                   806
 Fwd IAT Mean                   262
 Fwd IAT Min                    774
Bwd IAT Total                   346
 Bwd IAT Mean                   216
 Bwd IAT Std                    214
 Bwd IAT Min                    488
Fwd PSH Flags                    64
 Bwd PSH Flags                    0
 Fwd URG Flags                   17
 Bwd URG Flags                    0
 Bwd Packets/s                  385
 Min Packet Length              133
FIN Flag Count                   58
 RST Flag Count             

In [14]:
feature_importances_series = pd.Series(
    lgbm_gpu.feature_importances_, 
    index=X_train.columns
).sort_values(ascending=False)

# 2. Print the top 20 to decide on a cutoff

imp_features = feature_importances_series.head(20).index.tolist()
imp_features

[' Destination Port',
 ' Init_Win_bytes_backward',
 'Total Length of Fwd Packets',
 ' Flow IAT Min',
 ' Fwd IAT Min',
 'Init_Win_bytes_forward',
 ' Total Fwd Packets',
 ' Flow Duration',
 'Bwd Packet Length Max',
 ' Bwd IAT Min',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Mean',
 'Active Mean',
 ' Bwd Packets/s',
 'Flow Bytes/s',
 'Bwd IAT Total',
 ' Flow IAT Std',
 ' Flow Packets/s',
 ' Fwd IAT Mean',
 ' min_seg_size_forward']

In [15]:
# 1. Filter the importance scores to include ONLY the top 20 features
feature_importances_20 = feature_importances_series[imp_features]

# 2. Calculate the total importance score for these 20 features
# This sum will be used as the denominator for normalization
total_importance = feature_importances_20.sum()

# 3. Normalize the scores to get the percentage weight
# (Divide each score by the total sum and multiply by 100)
normalized_importance = (feature_importances_20 / total_importance) * 100

# 4. Sort and format the results for clear visualization
normalized_importance_sorted = normalized_importance.sort_values(ascending=False).round(2)

print("\n--- Normalized Feature Importance (Weight %) ---")
print(normalized_importance_sorted)


--- Normalized Feature Importance (Weight %) ---
 Destination Port              9.62
 Init_Win_bytes_backward       8.49
Total Length of Fwd Packets    7.79
 Flow IAT Min                  7.24
 Fwd IAT Min                   6.95
Init_Win_bytes_forward         6.51
 Total Fwd Packets             6.10
 Flow Duration                 5.83
Bwd Packet Length Max          4.62
 Bwd IAT Min                   4.38
 Fwd Packet Length Max         4.32
 Fwd Packet Length Mean        4.15
Active Mean                    3.82
 Bwd Packets/s                 3.46
Flow Bytes/s                   3.42
Bwd IAT Total                  3.11
 Flow IAT Std                  3.07
 Flow Packets/s                2.56
 Fwd IAT Mean                  2.35
 min_seg_size_forward          2.23
dtype: float64


In [16]:
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.impute import SimpleImputer
# imputer = SimpleImputer(strategy='mean')
# X_imputed = imputer.fit_transform(X_train)

# # Determine the number of columns (features) in your DataFrame
# num_columns = train_df.shape[1]

# # Selecting an appropriate K
# k = min(20, num_columns)  # Will adjust as needed

# # Initialize SelectKBest with the scoring function
# k_best = SelectKBest(score_func=f_classif, k=k)

# # Fit and transform the imputed data to select the top 10 features
# X_new = k_best.fit_transform(X_imputed, y_train)

# # Get the boolean mask of selected features
# selected_features_mask = k_best.get_support()
# elected_feature_names = X_train.columns[selected_features_mask]

# elected_feature_names

In [17]:
final_features = imp_features + [' Label']

train_df_preprocessed = train_final[final_features]
test_df_preprocessed = test_final[final_features]

In [18]:
config_data = {
    'project_name': "Network Anomaly Detection",
    'feature_engineering': {
        'selected_features': imp_features
    }
}

In [19]:
import yaml

with open('config.yaml',"a") as f:
    yaml.dump(config_data,f,default_flow_style=False)

In [20]:
train_df_preprocessed.to_parquet("final_df/train_df_preprocessed.parquet")
test_df_preprocessed.to_parquet("final_df/test_df_preprocessed.parquet")

In [21]:
test_df_preprocessed.head()

,Destination Port,Init_Win_bytes_backward,Total Length of Fwd Packets,Flow IAT Min,Fwd IAT Min,Init_Win_bytes_forward,Total Fwd Packets,Flow Duration,Bwd Packet Length Max,Bwd IAT Min,...,Fwd Packet Length Mean,Active Mean,Bwd Packets/s,Flow Bytes/s,Bwd IAT Total,Flow IAT Std,Flow Packets/s,Fwd IAT Mean,min_seg_size_forward,Label
0,22.038567,0.466102,0.967320,0.563636,1.471338,0.973298,1.2,4.920377,0.057551,14.155556,...,0.636364,89.310238,-0.000186,-0.008399,163.855894,1.387643,-0.001222,2.282925,0.0,1
1,22.038567,0.466102,-0.084967,1052.272727,-0.019108,-0.027186,-0.4,-0.007078,-0.001381,0.000000,...,-0.154791,0.000000,0.001287,-0.008479,0.000000,-0.007056,-0.000194,-0.000536,0.0,1
2,22.038567,1.004237,0.588235,1.945455,6.025478,0.247608,0.2,-0.000776,0.028085,36.888889,...,0.984029,0.000000,0.001715,0.012648,0.361417,0.008089,0.000327,0.015506,-1.0,1
3,4.851240,1.084746,-0.065359,1.236364,-0.019108,-0.027186,-0.4,-0.011814,0.000000,0.000000,...,-0.022113,0.000000,1.023615,1.188254,0.000000,-0.007056,0.713066,-0.000536,-1.0,1
4,22.038567,1.004237,0.588235,1.072727,4.726115,0.247608,0.2,-0.001352,0.028085,37.800000,...,0.984029,0.000000,0.001820,0.013811,0.342919,0.007299,0.000413,0.014669,-1.0,1


In [22]:
import joblib

joblib.dump(scaler,'final_df/fitted_robust_scaler.joblib')
joblib.dump(imputer,'final_df/fitted_robust_imputer.joblib')

['final_df/fitted_robust_imputer.joblib']